# P1-33 · Paper 1: analysis of the additional experiment W1 (faster-learning mapping)
**Runtime: CPU is enough (no GPU needed).** Run all (about 5-10 minutes).
Reads the per-image results of the 15 W1 runs and of the 15 standard E1 runs (HQB-UNet, HQB-cls, HQB-trig) in `MyDrive/P1_v2/runs/`.
Writes only aggregate numbers to `MyDrive/P1_v2/w1_analysis/` (`w1_results.csv`, `w1_weight_change.csv`, `w1_log.txt`). Nothing is
written to QSEG_FINAL. Analysis as prespecified in `PROTOCOL_w1_prespecified.txt`.

In [ ]:
# 1 · Drive and inputs
from google.colab import drive
drive.mount('/content/drive')
import os, json, numpy as np, pandas as pd
RUNS = '/content/drive/MyDrive/P1_v2/runs'; OUTD = '/content/drive/MyDrive/P1_v2/w1_analysis'
MIAS = '/content/drive/MyDrive/P1_v2/mias_ext'
os.makedirs(OUTD, exist_ok=True)
B = 10000; MARGIN = 0.01
MODELS = ('hq', 'hc', 'ht'); SEEDS = range(5)
missing = [f'{e}_{m}_seed{s}' for e in ('W1', 'E1') for m in MODELS for s in SEEDS if not os.path.exists(f'{RUNS}/{e}_{m}_seed{s}/test_per_image.csv')]
assert not missing, ('missing runs', missing)
def load(fname, exp):
    out = []
    for m in MODELS:
        for s in SEEDS:
            d = pd.read_csv(f'{RUNS}/{exp}_{m}_seed{s}/{fname}'); d['model'] = m; d['seed'] = s; d['exp'] = exp; out.append(d)
    return pd.concat(out, ignore_index=True)
T = pd.concat([load('test_per_image.csv', e) for e in ('W1', 'E1')]); T['cluster'] = T.patient_id.astype(str)
E = pd.concat([load('dmid_per_image.csv', e) for e in ('W1', 'E1')]); E['cluster'] = E.file_id.astype(str)
for X in (T, E):
    sw = [c for c in X.columns if c.startswith('drop_swap') and c[-1].isdigit()]
    X['drop_swapavg'] = X[sw].mean(axis=1)
print('test rows per run', T.groupby(['exp', 'model', 'seed']).size().unique(), '| DMID rows per run', E.groupby(['exp', 'model', 'seed']).size().unique(),
      '| patients', T.cluster.nunique())

In [ ]:
# 2 · Crossed hierarchical bootstrap (identical to the Paper 1 analysis: seeds and clusters resampled jointly, B = 10,000, rng(0))
def boot_stat(u, rng):
    seeds = np.sort(u.seed.unique()); cl = np.sort(u.cluster.unique())
    ci = {c: i for i, c in enumerate(cl)}; si = {s: i for i, s in enumerate(seeds)}
    S = np.zeros((len(cl), len(seeds))); N = np.zeros((len(cl), len(seeds)))
    for c, s, v in zip(u.cluster, u.seed, u.v):
        S[ci[c], si[s]] += v; N[ci[c], si[s]] += 1
    est = S.sum() / N.sum()
    wc = rng.multinomial(len(cl), np.ones(len(cl)) / len(cl), size=B)
    ws = rng.multinomial(len(seeds), np.ones(len(seeds)) / len(seeds), size=B)
    bs = np.einsum('bc,cs,bs->b', wc, S, ws) / np.einsum('bc,cs,bs->b', wc, N, ws)
    p = max(min(1.0, 2 * min((bs <= 0).mean(), (bs >= 0).mean())), 1.0 / B)
    return est, np.percentile(bs, 2.5), np.percentile(bs, 97.5), p
def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for r, i in enumerate(o):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1, run)
    return adj
def pair(X, ea, a, eb, b, col, seeds=SEEDS):
    x = X[(X.exp == ea) & (X.model == a) & X.seed.isin(seeds)][['file_id', 'cluster', 'seed', col]].rename(columns={col: 'a'})
    y = X[(X.exp == eb) & (X.model == b) & X.seed.isin(seeds)][['file_id', 'cluster', 'seed', col]].rename(columns={col: 'b'})
    m = x.merge(y, on=['file_id', 'cluster', 'seed']); assert len(m) == len(x) == len(y), (ea, a, eb, b, col)
    m['v'] = m.a - m.b
    return boot_stat(m.dropna(subset=['v']), np.random.default_rng(0))
def level(X, e, mod, col):
    t = X[(X.exp == e) & (X.model == mod)][['cluster', 'seed', col]].rename(columns={col: 'v'})
    return boot_stat(t.dropna(), np.random.default_rng(0))
def verdict(lo, hi):
    v = []
    if lo > 0 or hi < 0: v.append('difference')
    if -MARGIN < lo and hi < MARGIN: v.append('equivalent')
    return ', '.join(v) or 'inconclusive'
rows = []
def add(family, setname, col, label, r, seeds='0-4'):
    rows.append(dict(family=family, set=setname, metric=col, contrast=label, seeds=seeds, diff=r[0], ci_lo=r[1], ci_hi=r[2], p=r[3], verdict=verdict(r[1], r[2])))

In [ ]:
# 3 · Primary family (prespecified): W1 HQB-UNet - HQB-cls and HQB-UNet - HQB-trig, test Dice, Holm over the two
k0 = len(rows)
for b in ('hc', 'ht'):
    add('primary', 'test', 'dice', f'W1 hq - W1 {b}', pair(T, 'W1', 'hq', 'W1', b, 'dice'))
adj = holm([r['p'] for r in rows[k0:]])
for r, a in zip(rows[k0:], adj): r['p_holm'] = a
# secondary (descriptive): same contrasts on DMID; HQB-cls - HQB-trig; each W1 model minus its E1 counterpart; drops; per seed and leave-one-seed-out
for b in ('hc', 'ht'):
    add('secondary', 'dmid', 'dice', f'W1 hq - W1 {b}', pair(E, 'W1', 'hq', 'W1', b, 'dice'))
for X, nm in ((T, 'test'), (E, 'dmid')):
    add('secondary', nm, 'dice', 'W1 hc - W1 ht', pair(X, 'W1', 'hc', 'W1', 'ht', 'dice'))
    for m in MODELS:
        add('W1 minus E1', nm, 'dice', f'W1 {m} - E1 {m}', pair(X, 'W1', m, 'E1', m, 'dice'))
    for b in ('hc', 'ht'):
        add('E1 reference', nm, 'dice', f'E1 hq - E1 {b}', pair(X, 'E1', 'hq', 'E1', b, 'dice'))
    for col in ('drop_const', 'drop_swapavg'):
        for m in MODELS:
            add('code replacement', nm, col, f'W1 {m} level', level(X, 'W1', m, col))
            add('code replacement', nm, col, f'W1 {m} - E1 {m}', pair(X, 'W1', m, 'E1', m, col))
    for m in MODELS:
        add('levels', nm, 'dice', f'W1 {m} level', level(X, 'W1', m, 'dice'))
for b in ('hc', 'ht'):
    for s in SEEDS:
        add('per seed', 'test', 'dice', f'W1 hq - W1 {b}', pair(T, 'W1', 'hq', 'W1', b, 'dice', seeds=[s]), seeds=str(s))
        keep = [k for k in SEEDS if k != s]
        add('leave one seed out', 'test', 'dice', f'W1 hq - W1 {b}', pair(T, 'W1', 'hq', 'W1', b, 'dice', seeds=keep), seeds=f'without {s}')
print('done', len(rows))

In [ ]:
# 4 · mini-MIAS (descriptive): Dice recomputed from the saved probabilities against the expert masks in both orientations
# (dice_expert_asis is the verified orientation of the expert masks; the transposed version is a check) and against the proxy masks
mm = pd.read_csv(MIAS + '/manifest.csv'); mm = mm[mm.status == 'ok'].drop_duplicates('file_id').reset_index(drop=True); print('mini-MIAS images', len(mm))
from PIL import Image
def png(p): return np.array(Image.open(p).convert('L')) > 127
G = {'expert_asis': np.stack([png(f'{MIAS}/masks_expert/{f}.png') for f in mm.file_id])}
G['expert_transposed'] = np.transpose(G['expert_asis'], (0, 2, 1))
G['proxy'] = np.stack([png(f'{MIAS}/masks_proxy/{f}.png') for f in mm.file_id])
def dice(g, p):
    inter = (g & p).sum(axis=(1, 2)); s = g.sum(axis=(1, 2)) + p.sum(axis=(1, 2))
    return np.where(s == 0, 1.0, 2 * inter / np.maximum(s, 1))
mrows = []; avail = True
for e in ('W1', 'E1'):
    for m in MODELS:
        for s in SEEDS:
            f = f'{RUNS}/{e}_{m}_seed{s}/mias_prob_uint8.npz'
            if not os.path.exists(f): avail = False; continue
            z = np.load(f, allow_pickle=True); pos = {k: i for i, k in enumerate(z['file_id'])}
            if not all(k in pos for k in mm.file_id): avail = False; continue
            P = z['prob'][[pos[k] for k in mm.file_id]]; P = (P[..., 0] if P.ndim == 4 else P) >= 128
            d = pd.DataFrame({'file_id': mm.file_id, 'exp': e, 'model': m, 'seed': s})
            for k, g in G.items(): d['dice_' + k] = dice(g, P)
            mrows.append(d)
M = pd.concat(mrows, ignore_index=True)
M['cluster'] = ((M.file_id.str[3:].astype(int) + 1) // 2).astype(str)
print('mini-MIAS probabilities found for all E1 runs:', avail)
for col in ('dice_expert_transposed', 'dice_expert_asis', 'dice_proxy'):
    for b in ('hc', 'ht'):
        add('mini-MIAS', 'mias', col, f'W1 hq - W1 {b}', pair(M, 'W1', 'hq', 'W1', b, col))
    for m in MODELS:
        add('mini-MIAS', 'mias', col, f'W1 {m} level', level(M, 'W1', m, col))
        if avail: add('mini-MIAS', 'mias', col, f'W1 {m} - E1 {m}', pair(M, 'W1', m, 'E1', m, col))

In [ ]:
# 5 · Weight change of the mapping and its neighbours; run summaries
wrows = []
for m in MODELS:
    for s in SEEDS:
        d = f'{RUNS}/W1_{m}_seed{s}'
        wc = json.load(open(d + '/weight_change.json'))['weight_change']; r = json.load(open(d + '/run.json'))
        for layer, v in wc.items():
            wrows.append(dict(model=m, seed=s, layer=layer, **v, best_epoch=r['best_epoch'], epochs_run=r['epochs_run'], minutes=r['minutes'],
                              val_dice=r['val_dice'], test_dice=r['test_dice'], slot_max_change=r['slot_max_change']))
W = pd.DataFrame(wrows); W.to_csv(OUTD + '/w1_weight_change.csv', index=False)
R = pd.DataFrame(rows); R.to_csv(OUTD + '/w1_results.csv', index=False, float_format='%.6g')
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 300)
txt = R.drop(columns=['p']).round(4).to_string(index=False) + '\n\n' + \
      W.groupby(['model', 'layer'])[['max_abs_change', 'rel_change']].median().round(4).to_string()
open(OUTD + '/w1_log.txt', 'w').write(txt); print(txt)
print('\nW1 ANALYSIS DONE - results in MyDrive/P1_v2/w1_analysis/')